In [ ]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

# Die Onkologie-Station

Auf einer Onkologie-Station liegen fünf Patienten in nebeneinander liegenden Zimmern.
Bis auf einen  der Patienten hat jeder genau eine Zigaretten-Marke geraucht.
Der Patient, der nicht Zigarette geraucht hat, hat Pfeife geraucht.
Jeder Patient fährt genau ein Auto und ist
an genau einer Krebs-Art erkrankt.  Zusätzlich haben Sie die folgenden Informationen:
<ol>
<li> Im Zimmer neben Michael wird Camel geraucht. </li>
<li> Der Trabant-Fahrer raucht Ernte 23 und liegt im Zimmer neben dem 
      Zungen-Krebs Patienten. </li>
<li> Rolf liegt im letzten Zimmer und hat Kehlkopf-Krebs. </li>
<li> Der West-Raucher liegt im ersten Zimmer. </li>
<li> Der Mazda-Fahrer hat Zungen-Krebs und liegt neben dem Trabant-Fahrer. </li>
<li> Der Nissan-Fahrer liegt neben dem Zungen-Krebs Patient. </li>
<li> Rudolf wünscht sich Sterbe-Hilfe und liegt zwischen dem Camel-Raucher und dem Trabant-Fahrer. </li>
<li> Der Seat Fahrer hat morgen seinen letzten Geburtstag. </li>
<li> Der Luckies Raucher liegt neben dem Patienten mit Lungen-Krebs. </li>
<li> Der Camel Raucher liegt neben dem Patienten mit Darm-Krebs. </li>
<li> Der Nissan Fahrer liegt neben dem Mazda-Fahrer. </li>
<li> Der Mercedes-Fahrer raucht Pfeife und liegt neben dem Camel Raucher. </li>
<li> Jens liegt neben dem Luckies Raucher. </li>
<li> Der Hodenkrebs-Patient hat gestern seine Eier durchs Klo gespült. </li>
</ol>
Entwickeln Sie ein <em>F#</em>-Programm, das die folgenden Fragen beantwortet:
<ol> 
<li> Was raucht der Darmkrebs-Patient? </li>
<li> Was fährt Kurt für ein Auto? </li>
</ol>

## Importing the Necessary Notebooks

Our goal is to solve this puzzle by first coding it as a solvability problem of propositional logic and then to solve the resulting set of clauses using the algorithm of Davis and Putnam.  We import the notebook `07-Davis-Putnam-JW.ipynb`.  This notebook imports the notebook `04-CNF.ipynb`, which implements the function `normalize` that takes a formula and transforms it into a set of clauses.  The notebook `04-CNF.ipynb` in turn imports the parser for propositional logic.

In [ ]:
importNotebook "07-Davis-Putnam-JW.ipynb"

Using the parser and the function `normalize` we can implement the function `parseCNF`.  It takes one argument:
- `s` is a string representing a propositional formula.

It returns a set of clauses that is equivalent to the formula represented by `s`.

In [ ]:
let parseCNF (s: string) : CNF =
    normalize (parse s)

## Auxiliary Functions

The function `atMostOne` takes one argument:
- `V` is a set of propositional variables.

It returns a set of clauses that is true if and only if at most one of the variables from $V$ is true.

In [ ]:
let atMostOne (V: Set<string>) : CNF =
    set [ for p in V do
            for q in V do
                if p <> q then
                    set [ Neg p; Neg q ] ]

The function `var` takes two arguments:
- `f` is the name of a property, e.g. `"Japanese"`,
- `i` is the number of a room, i.e. $i \in\{1,2,3,4,5\}$.

It returns the string $f\langle i \rangle$, e.g. the call `var "Japanese" 2` returns the string `Japanese<2>`.

In [ ]:
let var (f: string) (i: int) : string =
    sprintf "%s<%d>" f i

In [ ]:
var "Japanese" 2

The function `flatten` takes one argument:
- `ListOfSets` is a list of sets.

It returns the union of these sets.

In [ ]:
let flatten (ListOfSets: Set<'T> list) : Set<'T> =
    Set.unionMany ListOfSets

In [ ]:
flatten [ set [1; 2; 3]; set [3; 4; 5] ]

The function `somewhere` takes one argument:
- `x` is the name of a property.

It returns a clause that specifies that the person with property $x$ has to stay in one of the rooms from $1$ to $5$.

In [ ]:
let somewhere (x: string) : Clause =
    set [ for i in 1 .. 5 -> Pos (var x i) ]

In [ ]:
somewhere "a"

The function `atMostOneAt` takes two arguments:
- `S` is a set of mutually exclusive properties,
- `i` is the number of a room.

It returns a set of clauses that specifies that the person staying in room number $i$ has at most one of the properties from the set $S$.  For example, if 
$S = \{\texttt{"Camel"}, \texttt{"Ernte"}, \texttt{"West"}, \texttt{"Luckies"}, \texttt{"Pfeife"}\}$, 
then $\texttt{atMostOneAt}(S, 3)$ specifies that the patient in room number 3 smokes at most one of these brands.

In [ ]:
let atMostOneAt (S: Set<string>) (i: int) : CNF =
    atMostOne (Set.map (fun x -> var x i) S)

In [ ]:
atMostOneAt (set [ "A"; "B"; "C" ]) 1

The function `onePerRoom` takes one argument:
- `S` is a set of five mutually exclusive properties.

It returns a set of clauses that expresses that for every property from $S$ there is a room where the patient with this property stays.  Furthermore, the set of clauses contains clauses that express that no patient has two of the properties from $S$.

In [ ]:
let onePerRoom (S: Set<string>) : CNF =
    set [ for x in S -> somewhere x ] + flatten [ for i in 1 .. 5 -> atMostOneAt S i ]

In [ ]:
onePerRoom (set [ "A"; "B"; "C"; "D"; "E" ])

The function `sameRoom` takes two arguments:
- `a` is a property,
- `b` is a property.

It returns a set of clauses that specifies that if the patient in room number $i$ has the property $a$, then he also has the property $b$ and vice versa.

In [ ]:
let sameRoom (a: string) (b: string) : CNF =
    flatten [ for i in 1 .. 5 -> parseCNF (sprintf "%s ↔ %s" (var a i) (var b i)) ]

In [ ]:
sameRoom "Luckies" "Darm"

The function `differentRoom` takes two arguments:
- `a` is a property,
- `b` is a property.

It returns a set of clauses that specifies that the patients with properties $a$ and $b$ stay in different rooms. 

In [ ]:
let differentRoom (a: string) (b: string) : CNF =
    failwith "your code here"

In [ ]:
differentRoom "A" "B"

The function `nextTo` takes two arguments:
- `a` is a property,
- `b` is a property.

It returns a set of clauses that specifies that the patients with properties $a$ and $b$ stay in neighbouring rooms.

In [ ]:
let nextTo (a: string) (b: string) : CNF =
    parseCNF (sprintf "%s → %s" (var a 1) (var b 2))
    + flatten [ for i in 2 .. 4 ->
                  parseCNF (sprintf "%s → %s ∨ %s" (var a i) (var b (i-1)) (var b (i+1))) ]
    + parseCNF (sprintf "%s → %s" (var a 5) (var b 4))

In [ ]:
nextTo "A" "B"

In [ ]:
let Brands  = set [ "Camel"; "Ernte"; "West"; "Luckies"; "Pfeife" ]
let Cars    = set [ "Trabant"; "Mazda"; "Nissan"; "Seat"; "Mercedes" ]
let Cancers = set [ "Zunge"; "Kehlkopf"; "Lunge"; "Darm"; "Hoden" ]
let Names   = set [ "Michael"; "Rolf"; "Rudolf"; "Jens"; "Kurt" ]

The function `allClauses` takes no arguments.  It returns a set of clauses describing the problem.

In [ ]:
let allClauses () : CNF =
    let mutable Clauses = onePerRoom Brands
    Clauses <- Clauses + onePerRoom Cars
    Clauses <- Clauses + onePerRoom Cancers
    Clauses <- Clauses + onePerRoom Names
    // Im Zimmer neben Michael wird Camel geraucht. 
    Clauses <- Clauses + failwith "your code here"
    // Der Trabant-Fahrer raucht Ernte 23 und liegt im Zimmer neben dem 
    // Zungen-Krebs Patienten. 
    Clauses <- Clauses + failwith "your code here"
    // Rolf liegt im letzten Zimmer und hat Kehlkopf-Krebs.
    Clauses <- Clauses + failwith "your code here"
    // Der West-Raucher liegt im ersten Zimmer. 
    Clauses <- Clauses + failwith "your code here"
    // Der Mazda-Fahrer hat Zungen-Krebs und liegt neben dem Trabant-Fahrer. 
    Clauses <- Clauses + failwith "your code here"
    // Der Nissan-Fahrer liegt neben dem Zungen-Krebs Patient. 
    Clauses <- Clauses + failwith "your code here"
    // Rudolf wünscht sich Sterbe-Hilfe und liegt zwischen dem Camel-Raucher und dem Trabant-Fahrer. 
    Clauses <- Clauses + failwith "your code here"
    // Der Luckies Raucher liegt neben dem Patienten mit Lungen-Krebs. 
    Clauses <- Clauses + failwith "your code here"
    // Der Camel Raucher liegt neben dem Patienten mit Darm-Krebs. 
    Clauses <- Clauses + failwith "your code here"
    // Der Nissan Fahrer liegt neben dem Mazda-Fahrer. 
    Clauses <- Clauses + failwith "your code here"
    // Der Mercedes-Fahrer raucht Pfeife und liegt neben dem Camel Raucher. 
    Clauses <- Clauses + failwith "your code here"
    // Jens liegt neben dem Luckies Raucher. 
    Clauses <- Clauses + failwith "your code here"
    // Der Hodenkrebs-Patient hat gestern seine Eier durchs Klo gespült.
    Clauses

In [ ]:
let Clauses = allClauses ()
printfn "%s" (cnfToString Clauses)

I have got 322 clauses.

In [ ]:
Clauses.Count

Solving the problem takes less than a second.

In [ ]:
#!time
let Solution = solve Clauses

## Pretty Printing the Solution

The function `extractAssignment` takes one argument:
- `Solution` is a set of unit clauses representing a solution of the puzzle.

It returns a map that assigns to every property the number of the room of the patient having this property.  Only the positive literals are relevant.  The function `arb` is defined in the notebook `07-Davis-Putnam-JW.ipynb`.

In [ ]:
let extractAssignment (Solution: CNF) : Map<string, int> =
    Map [ for Unit in Solution do
            match arb Unit with
            | Pos v -> let k = v.IndexOf '<'
                       (v.Substring(0, k), int (v.Substring(k + 1, v.Length - k - 2)))
            | Neg _ -> () ]

The function `showHTML` takes one argument:
- `Assignment` is a map assigning room numbers to properties.

It returns an HTML table that shows the solution.  The table has one row for every room.

In [ ]:
let showHTML (Assignment: Map<string, int>) =
    let sb = System.Text.StringBuilder()
    sb.Append("<table style=\"border:2px solid blue\">\n<tr>") |> ignore
    for name in [ "Room"; "Brands"; "Cars"; "Cancers"; "Names" ] do
        sb.Append("<th style=\"color:gold; background-color:blue\">" + name + "</th>") |> ignore
    sb.Append("</tr>\n") |> ignore
    for room in 1 .. 5 do
        sb.Append("<tr><td style=\"border:1px solid green\">" + string room + "</td>") |> ignore
        for Class in [ Brands; Cars; Cancers; Names ] do
            for x in Class do
                if Assignment[x] = room then
                    sb.Append("<td style=\"border:1px solid green\">" + x + "</td>") |> ignore
        sb.Append("</tr>\n") |> ignore
    sb.Append("</table>") |> ignore
    HTML(sb.ToString())

In [ ]:
showHTML (extractAssignment Solution)

## Checking the Uniqueness of the Solution

The function `negateSolution` takes one argument:
- `UnitClauses` is a set of unit clauses.

It returns a clause that is the negation of the conjunction of these unit clauses.

In [ ]:
let negateSolution (UnitClauses: CNF) : Clause =
    Set.map (fun Unit -> complement (arb Unit)) UnitClauses

In [ ]:
negateSolution (set [ set [ Pos "a" ]; set [ Neg "b" ] ])

The function `checkUniqueness` takes two arguments:
- `Solution` is a set of unit clauses representing a solution of `Clauses`,
- `Clauses` is a set of clauses.

It checks whether `Solution` is the only solution of `Clauses`.  To this end, the negation of `Solution` is added to `Clauses`.  If the resulting set of clauses is unsatisfiable, the solution is unique.

In [ ]:
let checkUniqueness (Solution: CNF) (Clauses: CNF) : unit =
    let negation    = negateSolution Solution
    let alternative = solve (Clauses.Add negation)
    if alternative = set [ Set.empty ] then
        printfn "Well done: The solution is unique!"
    else
        printfn "ERROR: The solution is not unique!"

In [ ]:
checkUniqueness Solution Clauses